# Chapter 7 -- 第二个分类器：k 近邻（kNN）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chaosfrey-arch/news-classification-tutorial/blob/main/chapter07_knn.ipynb)

**本章目标：** 理解 kNN 的原理，解决高维问题，用交叉验证选最优 k。

**预计时间：** 60 分钟

> 延伸阅读：[维度灾难（Wikipedia）](https://en.wikipedia.org/wiki/Curse_of_dimensionality)

## 7.1 kNN 的直觉：物以类聚

**核心思想：** 要判断一个新样本的类别，就看它的 k 个最近邻居是什么类别。

生活类比：
- 你搬到一个新城市，想知道这个小区是什么风格
- 看看附近 5 家餐馆（k=5）：3 家川菜、1 家粤菜、1 家火锅
- 多数是川菜 → 判断这是个川菜风格的街区

在机器学习里：
- 每篇文章是空间里的一个**点**（用数字向量表示）
- 相似的文章距离近
- 新文章的类别 = 距它最近的 k 篇已知文章中，出现最多的类别

## 7.2 维度灾难（Curse of Dimensionality）

我们的 TF-IDF 矩阵有 20,000 维（每篇文章是 20,000 维空间里的一个点）。

问题：**在高维空间里，「距离」的概念几乎失效了。**

直觉解释：
- 在 1D（一条线）上：随机放 100 个点，它们分布比较均匀，总能找到真正近的邻居
- 在 2D（一个平面）上：需要更多的点才能覆盖空间
- 在 20000D 上：所有点之间的距离变得几乎一样远！
  就像在宇宙里找邻居，每颗星离你都差不多远

**结果：** kNN 在原始 TF-IDF 空间上效果会很差，必须先降维。

## 7.3 解决方法：SVD 降维（Latent Semantic Analysis）

**Truncated SVD（截断奇异值分解）** = 从高维数据里找出最重要的「方向」，投影到低维空间。

类比：
- 你有一张 4K 分辨率的图（高维）
- 把它压缩成 720P（低维）
- 大部分视觉信息还在，但数据量少多了

我们把 20,000 维压缩到 **200 维**：
- 保留了约 15.6% 的方差信息
- 去掉了噪声
- 降维后的空间里，相似文章的距离更有意义

**LSA（Latent Semantic Analysis）** = 用 SVD 对文本 TF-IDF 矩阵降维，这是信息检索领域的经典方法。

In [ ]:
import pandas as pd, numpy as np, re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

def clean_text(t):
    t = str(t).lower()
    t = re.sub(r'http\S+|www\S+|https\S+', '', t)
    t = re.sub(r'[^a-z\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

df = pd.read_csv('dataset.csv').dropna(subset=['Class']).copy()
df['Description'] = df['Description'].fillna('')
df['text'] = (df['Title'] + ' ' + df['Description']).apply(clean_text)
le = LabelEncoder()
y = le.fit_transform(df['Class'])
X_train, X_test, y_train, y_test = train_test_split(
    df['text'].values, y, test_size=0.3, random_state=42, stratify=y)

tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1,2), sublinear_tf=True, min_df=2)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)
print('数据准备完毕！')

In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer

# 步骤 1：SVD 降维到 200 维
svd = TruncatedSVD(n_components=200, random_state=42)

# 步骤 2：L2 归一化（让每个向量长度为 1）
# 好处：这样欧氏距离等价于余弦相似度，更适合文本
normalizer = Normalizer(copy=False)

# 注意：只在训练集上 fit，测试集只 transform
X_train_lsa = normalizer.fit_transform(svd.fit_transform(X_train_tfidf))
X_test_lsa  = normalizer.transform(svd.transform(X_test_tfidf))

print(f'降维前：{X_train_tfidf.shape[1]:,} 维')
print(f'降维后：{X_train_lsa.shape[1]} 维')
print(f'保留方差：{svd.explained_variance_ratio_.sum():.1%}')
print(f'\n降维后训练集形状：{X_train_lsa.shape}')

# 保存降维器（后面 predict 函数需要用）
import joblib
joblib.dump(svd, 'knn_svd.joblib')
joblib.dump(normalizer, 'knn_normalizer.joblib')

## 7.4 怎么选 k：交叉验证

**问题：** k 越大越好还是越小越好？
- k=1：只看最近的 1 个邻居，容易受噪声影响（过拟合）
- k=100：看太多邻居，把离得很远的文章也纳入，判断变模糊（欠拟合）

**交叉验证（Cross-Validation）：**
把训练集再分成 5 份，轮流用 1 份验证、4 份训练，重复 5 次，取平均成绩。

```
训练集 → 分成5份
第1轮：[验证|训练|训练|训练|训练] → 得到 F1_1
第2轮：[训练|验证|训练|训练|训练] → 得到 F1_2
...（共5轮）
平均 F1 = (F1_1 + ... + F1_5) / 5
```

**为什么不直接用测试集选 k？** 那叫「数据泄露」——你根据考试结果调整了学习策略，考试就失去了意义。

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
import matplotlib.pyplot as plt

k_values = [1, 3, 5, 7, 9, 11]
cv_scores = []

# 5 折分层交叉验证
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('正在进行交叉验证，请耐心等待（约需 10-20 分钟）...')
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean', n_jobs=-1)
    scores = cross_val_score(knn, X_train_lsa, y_train, cv=skf,
                             scoring='f1_weighted', n_jobs=-1)
    cv_scores.append(scores.mean())
    print(f'  k={k:2d} -> CV F1 = {scores.mean():.4f} (+-{scores.std():.4f})')

best_k = k_values[int(np.argmax(cv_scores))]
print(f'\n最优 k = {best_k}（CV F1 = {max(cv_scores):.4f}）')

In [ ]:
# 画 k vs F1 图
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(k_values, cv_scores, marker='o', linewidth=2, markersize=9, color='steelblue')
ax.axvline(best_k, color='red', linestyle='--', label=f'Best k={best_k}')
ax.set_title('kNN: Cross-Validation F1 vs. k', fontsize=14, fontweight='bold')
ax.set_xlabel('k (Number of Neighbours)', fontsize=12)
ax.set_ylabel('Weighted F1-Score (5-fold CV)', fontsize=12)
ax.set_xticks(k_values)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('观察：F1 随 k 增大而单调增加（k=11 最优），说明更多邻居更稳定。')

In [ ]:
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

# 用最优 k 训练最终模型
knn_model = KNeighborsClassifier(n_neighbors=best_k, metric='euclidean', n_jobs=-1)
knn_model.fit(X_train_lsa, y_train)

y_pred_knn = knn_model.predict(X_test_lsa)

print(f'=== kNN (k={best_k}) 测试集结果 ===')
print(f'  Accuracy  : {accuracy_score(y_test, y_pred_knn):.4f}')
print(f'  F1-Score  : {f1_score(y_test, y_pred_knn, average="weighted"):.4f}')
print(f'  Precision : {precision_score(y_test, y_pred_knn, average="weighted"):.4f}')
print(f'  Recall    : {recall_score(y_test, y_pred_knn, average="weighted"):.4f}')
print()
print(classification_report(y_test, y_pred_knn, target_names=le.classes_))

cm = confusion_matrix(y_test, y_pred_knn)
fig, ax = plt.subplots(figsize=(7,5))
ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(ax=ax, cmap='Greens', colorbar=False)
ax.set_title(f'kNN (k={best_k}) -- Confusion Matrix')
plt.tight_layout()
plt.show()

joblib.dump(knn_model, 'knn_model.joblib')
print('kNN 模型已保存！')

## 总结

**kNN 结果：Accuracy ≈ 84%（低于 Naive Bayes）**

原因：LSA 只保留了 15.6% 的方差，丢失了大量信息。

| 概念 | 含义 |
|------|------|
| kNN | 找 k 个最近邻居，用多数类别投票 |
| 维度灾难 | 高维空间里距离失效 |
| SVD/LSA | 降维，保留主要信息 |
| L2 归一化 | 让欧氏距离等价于余弦相似度 |
| 交叉验证 | 在训练集内评估，避免数据泄露 |

**下一章 →** Chapter 8：神经网络基础